In [ ]:
import os
import pandas as pd
from itertools import combinations

BASE_PATH = "/home/zstoimchev/Documents/Thesis2026/codespace/previous_work/data/parquets"

datasets = {
    "CIC-IDS2017": "CIC-IDS2017/Network-Flows",
    "CIC-IDS2018": "CIC-IDS2018/NF-CSE-CIC-IDS2018-V2.parquet",
    "CIC-IDS-Collection": "CIC-IDS-Collection/cic_collection.parquet",
    "UNSW-NB15": "UNSW-NB15/Network-Flows",
    "UNSW-NB15v2": "UNSW-NB15v2/NF-UNSW-NB15.parquet"
}

# --- Normalization function ---
def normalize(col):
    return (
        col.strip()
        .lower()
        .replace(" ", "_")
        .replace("/", "_")
        .replace(".", "_")
    )

# --- Load columns ---
def get_columns(path):
    if os.path.isdir(path):
        files = [f for f in os.listdir(path) if f.endswith(".parquet")]
        if not files:
            return set(), set()
        file_path = os.path.join(path, files[0])
    else:
        file_path = path

    df = pd.read_parquet(file_path)

    raw_cols = set(df.columns)
    norm_cols = set(normalize(c) for c in df.columns)

    return raw_cols, norm_cols

# --- Extract ---
raw_map = {}
norm_map = {}

print("=== DATASET COLUMN COUNTS ===\n")

for name, rel_path in datasets.items():
    full_path = os.path.join(BASE_PATH, rel_path)

    raw, norm = get_columns(full_path)

    raw_map[name] = raw
    norm_map[name] = norm

    print(f"{name}:")
    print(f"  Raw columns: {len(raw)}")
    print(f"  Normalized columns: {len(norm)}\n")

# --- Save full column lists ---
with open("PARQUET_COLUMNS_NORMALIZED.txt", "w") as f:
    for name, cols in norm_map.items():
        f.write(f"\n===== {name} ({len(cols)} columns) =====\n")
        for c in sorted(cols):
            f.write(c + "\n")

# --- Pairwise intersections ---
with open("PARQUET_INTERSECTIONS_NORMALIZED.txt", "w") as f:
    for (n1, c1), (n2, c2) in combinations(norm_map.items(), 2):
        inter = c1 & c2
        f.write(f"{n1} ∩ {n2} = {len(inter)}\n")

# --- Global intersection ---
common_all = set.intersection(*norm_map.values())

with open("PARQUET_COMMON_ALL_NORMALIZED.txt", "w") as f:
    f.write(f"Common columns across ALL datasets ({len(common_all)}):\n")
    for c in sorted(common_all):
        f.write(c + "\n")

print("\nAnalysis complete.")
print("Generated:")
print("- PARQUET_COLUMNS_NORMALIZED.txt")
print("- PARQUET_INTERSECTIONS_NORMALIZED.txt")
print("- PARQUET_COMMON_ALL_NORMALIZED.txt")
